# Extract data from CERCA raw files

In [74]:
from docx import Document
import pandas as pd
from google.cloud import bigquery
import requests
from tqdm import tqdm

In [2]:
interest_centers = ['BETA', 'CREAF', 'ICN2', 'ISGlobal', 'ResearchMar']
file_path = '../data/external/5_Bibliometria_SIRIS_081025/'

## Extract DOI list

**Each file has a different format so we go center by center**

### BETA

In [3]:
center_name = 'BETA/'
file_name = 'LIST OF SCIENTIFIC PUBLICATIONS_BETA'

In [4]:
doc = Document(file_path + 'BM_' + center_name + file_name + '.docx')
pubs = [para.text for para in doc.paragraphs]
DOI_tmp = [pub.split('DOI:', 1)[1].strip() for pub in pubs  if 'DOI:' in pub]
DOI_BETA = [(DOI.split('https://doi.org/', 1)[1].strip() if 'https://doi.org/' in DOI else DOI) for DOI in DOI_tmp]
df_BETA = pd.DataFrame(DOI_BETA, columns = ['DOI'])
df_BETA['Center'] = 'BETA'
df_BETA

,DOI,Center
0,10.1016/j.scitotenv.2023.168824,BETA
1,10.23818/limn.43.07,BETA
2,10.1016/j.aquatox.2024.106843,BETA
3,10.3390/agronomy14050935,BETA
4,10.32347/2077-3455.2024.68.215-227,BETA
...,...,...
63,10.1016/j.bcab.2021.102114,BETA
64,10.1016/j.sajb.2021.06.035,BETA
65,10.1002/ecy.3614,BETA
66,10.3390/ijms222011277,BETA


### CREAF

In [5]:
center_name = 'CREAF/'
file_name = 'Publicacions CREAF 2021-2024'

In [6]:
df = pd.read_excel(file_path + 'BM_' + center_name + file_name + '.xlsx', skiprows = 5)
df_CREAF = df[['doi']].rename(columns = {'doi': 'DOI'})
df_CREAF['Center'] = 'CREAF'
df_CREAF

,DOI,Center
0,10.1038/s43705-021-00073-5,CREAF
1,10.7325/galemys.2022.n5,CREAF
2,10.1109/IGARSS53475.2024.10640524,CREAF
3,10.1109/IGARSS53475.2024.10641964,CREAF
4,10.1109/MetroAgriFor63043.2024.10948835,CREAF
...,...,...
928,10.18601/01245996.v24n47.12,CREAF
929,10.1038/s43247-021-00229-0,CREAF
930,10.34133/2022/9764982,CREAF
931,10.34133/remotesensing.0085,CREAF


### ICN2

In [7]:
center_name = 'ICN2/'
file_name = 'ICN2_DOIs_Pubs2021-2024_CERCA2025'

In [8]:
df_ICN2 = pd.read_excel(file_path + 'BM_' + center_name + file_name + '.xlsx').rename(columns = {'Doi': 'DOI'})
df_ICN2['Center'] = 'ICN2'
df_ICN2

,DOI,Center
0,10.59277/ROMJIST.2024.2.09,ICN2
1,10.1002/ece2.12,ICN2
2,10.1093/mam/ozae044.520,ICN2
3,10.1103/physrevlett.132.266301,ICN2
4,10.7203/metode.15.27225,ICN2
...,...,...
823,10.1103/PhysRevB.108.054524,ICN2
824,10.1021/acsaem.1c02919,ICN2
825,10.1016/j.synthmet.2021.116844,ICN2
826,10.1038/s42254-021-00318-1,ICN2


### ISGlobal

In [9]:
center_name = 'ISGlobal/'
file_name = 'ISGlobal_DOIs Publications_2021-2024'

In [10]:
df = pd.read_excel(file_path + 'BM_' + center_name + file_name + '.xlsx', skiprows = 1)
df_ISGlobal = df[['DOI']]
df_ISGlobal['Center'] = 'ISGlobal'
df_ISGlobal

/tmp/ipykernel_46646/4287051607.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_ISGlobal['Center'] = 'ISGlobal'


,DOI,Center
0,10.1038/s41380-019-0558-2,ISGlobal
1,10.1016/j.edumed.2019.09.004,ISGlobal
2,10.1038/ng.2238,ISGlobal
3,10.1111/all.14422,ISGlobal
4,10.1016/j.nrl.2020.02.006,ISGlobal
...,...,...
753,10.3390/ijms222413363,ISGlobal
754,10.3390/pathogens10121588,ISGlobal
755,10.1371/journal.pntd.0009954,ISGlobal
756,10.1136/bmjopen-2021-052897,ISGlobal


### ResearchMar

In [11]:
center_name = 'ResearchMar/'
file_name = 'HMRIB_CERCA_DOIs21-24_VF'

In [12]:
df_ResearchMar = pd.read_excel(file_path + 'BM_' + center_name + file_name + '.xlsx', header = None)[0:-1].rename(columns = {0: 'DOI'})
df_ResearchMar['Center'] = 'df_ResearchMar'
df_ResearchMar

,DOI,Center
0,10.1001/jama.2021.15255,df_ResearchMar
1,10.1001/jama.2022.1645,df_ResearchMar
2,10.1001/jamacardio.2021.4690,df_ResearchMar
3,10.1001/jamacardio.2022.1988,df_ResearchMar
4,10.1001/jamadermatol.2022.0434,df_ResearchMar
...,...,...
5346,10.7759/cureus.13183,df_ResearchMar
5347,10.7759/cureus.16472,df_ResearchMar
5348,10.7759/cureus.40708,df_ResearchMar
5349,10.7759/cureus.62509,df_ResearchMar


## Check which publications are not in OA using DOI

In [13]:
PROJECT_ID = 'siris-datasets'
DATASET_ID = 'openalex'

def bg_query(query):
    client = bigquery.Client(project=PROJECT_ID)
    df = client.query(query)
    return df.to_dataframe()

In [ ]:
df_centers = pd.concat([df_BETA, df_CREAF, df_ICN2, df_ISGlobal, df_ResearchMar], ignore_index = True)

in_query = str(tuple(df_centers.DOI.tolist()))
in_query = in_query.replace(',)', ')')

sql = """select w.id, w.DOI
         FROM """ + PROJECT_ID + "." + DATASET_ID + ".works" + " w" """
         WHERE w.DOI in {}""".format(in_query)
df_query = bg_query(sql)

df_query

/home/siris/2025CERCA01/2025CERCA01_env/lib/python3.10/site-packages/google/auth/_default.py:108: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)
/home/siris/2025CERCA01/2025CERCA01_env/lib/python3.10/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,id,DOI
0,4304190627,10.1016/j.redare.2021.05.021
1,3182940708,10.1038/s41467-021-24319-x
2,3133860003,10.1016/j.cmpb.2021.106042
3,3156031087,10.1080/17538157.2021.1902332
4,4385513668,10.1016/j.jenvman.2023.118627
...,...,...
6453,3118361619,10.1016/j.arbr.2021.05.019
6454,4317234381,10.1001/jamadermatol.2022.5991
6455,4376114109,10.1016/j.medine.2023.04.005
6456,4223488237,10.1016/j.ijsu.2022.106611


In [79]:
missing_DOI = df_centers[~df_centers["DOI"].isin(df_query["DOI"])]
print('The number of missing DOIs is:', missing_DOI.DOI.nunique())
missing_DOI

The number of missing DOIs is: 482


,DOI,Center
16,10.7818/ECOS.2684,BETA
19,10.3390/su16010238.,BETA
27,10.5683/SP3/BIDMCI,BETA
70,10.1109/IGARSS53475.2024.10640524,CREAF
71,10.1109/IGARSS53475.2024.10641964,CREAF
...,...,...
7925,10.7554/eLife.80556,df_ResearchMar
7926,10.7554/eLife.81067,df_ResearchMar
7927,10.7554/eLife.81067,df_ResearchMar
7928,10.7554/eLife.85893,df_ResearchMar


**Lets try to see if we can get them from Open aire**

In [ ]:
def extract_value(field):
    if isinstance(field, list):
        return field[0].get("$", None) if field else None
    return field.get("$", None)

BATCH_SIZE = 25
SLEEP_BETWEEN_REQUESTS = 1.0
BASE_URL = "https://api.openaire.eu/search/publications"

records = pd.DataFrame(columns=["id", "DOI"])
not_found_dois = []

for i in tqdm(range(0, len(missing_DOI.DOI), BATCH_SIZE)):
    batch = missing_DOI.DOI[i:i+BATCH_SIZE]
    found_in_batch = []

    doi_param = ",".join(batch)
    params = {"doi": doi_param, "format": "json", "size": BATCH_SIZE}

    try:
        response = requests.get(BASE_URL, params=params)
        response.raise_for_status()
        openaire_results = response.json()

        results = openaire_results.get("response", {}).get("results", {}).get("result", [])
        if not isinstance(results, list):
            results = [results]

        for result in results:
            metadata = result.get("metadata", {}).get("oaf:entity", {}).get("oaf:result", {})
            doi_found = extract_value(metadata.get("pid", {}))
            id = extract_value(metadata.get("originalId", {}))

            if doi_found:
                records = pd.concat(
                    [records, pd.DataFrame([{"id": id, "DOI": doi_found}])],
                    ignore_index=True
                )
                found_in_batch.append(doi_found)

        # Compare with original DOIs in the batch
        batch_not_found = [d for d in batch if d not in found_in_batch]
        not_found_dois.extend(batch_not_found)

    except Exception as e:
        print(f"⚠️ Error processing batch {i//BATCH_SIZE + 1}: {e}")

print('Number of DOIs not found:', len(not_found_dois))
not_found_dois



 57%|█████▋    | 13/23 [00:49<00:30,  3.02s/it]